# 01. Auditoria de insumos del proyecto

Este notebook audita los archivos base del proyecto antes de continuar con la fase de limpieza, integracion y modelado.

## Objetivos
- identificar los archivos disponibles en `BASE_DE_DATOS`
- clasificar que archivos son de entrada, intermedios o finales
- revisar estructura, cobertura temporal, columnas, nulos y duplicados
- comparar las bases existentes para detectar diferencias importantes
- dejar una lectura tecnica clara para decidir con que archivos seguir trabajando

## Archivos esperados
- `BASE_DE_DATOS/ENTRADA/agronet_cafe.xlsx`
- `BASE_DE_DATOS/ENTRADA/datos_seguro_cafe_2000_2025.csv`
- `BASE_DE_DATOS/INTERMEDIAS/base_datos_seguro_cafe.csv`
- `BASE_DE_DATOS/FINALES/cafe_clean.csv`
- `BASE_DE_DATOS/VARIABLES DEL MODELO - SEGURO AGRICOLA.txt` o similar

## Resultado esperado
Al finalizar este notebook debemos poder responder:

1. Que archivos tenemos realmente.
2. Que rol cumple cada uno dentro del pipeline.
3. Que diferencias estructurales existen entre las bases.
4. Que base conviene usar como punto de partida para el siguiente notebook.


## Instrucciones de uso

Este notebook esta pensado para ejecutarse desde la carpeta `MODELOS` del proyecto. Si se ejecuta desde otra ubicacion, el bloque de configuracion intenta encontrar automaticamente la raiz del proyecto buscando la carpeta `BASE_DE_DATOS`.

Si alguna ruta no existe, el notebook lo reporta explicitamente para no seguir trabajando sobre supuestos incorrectos.


In [19]:
from pathlib import Path
import json
import math
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 120)


def find_project_root(start: Path) -> Path:
    """Busca la raiz del proyecto subiendo por los padres hasta encontrar BASE_DE_DATOS."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'

PATHS = {
    'entrada_agronet': BASE_DATOS / 'ENTRADA' / 'agronet_cafe.xlsx',
    'entrada_clima_satelite': BASE_DATOS / 'ENTRADA' / 'datos_seguro_cafe_2000_2025.csv',
    'intermedia_integrada': BASE_DATOS / 'INTERMEDIAS' / 'base_datos_seguro_cafe.csv',
    'final_limpia': BASE_DATOS / 'FINALES' / 'cafe_clean.csv',
}

txt_candidates = sorted(BASE_DATOS.glob('*.txt'))
PATHS['documentacion_txt'] = txt_candidates[0] if txt_candidates else None

print(f'Ruta actual: {CURRENT_DIR}')
print(f'Raiz detectada del proyecto: {PROJECT_ROOT}')
print(f'Carpeta BASE_DE_DATOS: {BASE_DATOS}')


Ruta actual: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\MODELOS
Raiz detectada del proyecto: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO
Carpeta BASE_DE_DATOS: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS


In [20]:
existencia = []
for nombre, ruta in PATHS.items():
    existencia.append({
        'recurso': nombre,
        'ruta': str(ruta) if ruta is not None else None,
        'existe': bool(ruta and ruta.exists())
    })

df_existencia = pd.DataFrame(existencia)
df_existencia


,recurso,ruta,existe
0,entrada_agronet,C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\ENTRADA\agronet_cafe.xlsx,True
1,entrada_clima_satelite,C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\ENTRADA\datos_seguro_cafe_2000_2025.csv,True
2,intermedia_integrada,C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\INTERMEDIAS\base_datos_seguro_cafe.csv,True
3,final_limpia,C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\FINALES\cafe_clean.csv,True
4,documentacion_txt,C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\VARIABLES DEL MODELO - SEGURO AGRÍCOLA.txt,True


## Inventario general de la carpeta `BASE_DE_DATOS`

Este bloque lista el contenido de la carpeta para validar que la organizacion actual coincide con el flujo del proyecto.


In [21]:
inventario = []
for path in sorted(BASE_DATOS.rglob('*')):
    inventario.append({
        'tipo': 'carpeta' if path.is_dir() else 'archivo',
        'ruta_relativa': str(path.relative_to(BASE_DATOS)),
        'extension': path.suffix.lower() if path.is_file() else '',
        'tamano_bytes': path.stat().st_size if path.is_file() else None,
    })

df_inventario = pd.DataFrame(inventario)
df_inventario


,tipo,ruta_relativa,extension,tamano_bytes
0,carpeta,ENTRADA,,NaN
1,archivo,ENTRADA\agronet_cafe.xlsx,.xlsx,101880.0
2,archivo,ENTRADA\datos_seguro_cafe_2000_2025.csv,.csv,323372.0
3,carpeta,FINALES,,NaN
4,archivo,FINALES\cafe_clean.csv,.csv,200618.0
5,carpeta,INTERMEDIAS,,NaN
6,archivo,INTERMEDIAS\base_datos_seguro_cafe.csv,.csv,253077.0
7,archivo,VARIABLES DEL MODELO - SEGURO AGRÍCOLA.txt,.txt,13513.0


## Funciones auxiliares de perfilado

Las siguientes funciones resumen cada archivo de forma compacta para no tener que inspeccionarlo manualmente cada vez.


In [22]:
def summarize_csv(path: Path, sep: str = ';', n_preview: int = 3):
    df = pd.read_csv(path, sep=sep)
    summary = {
        'archivo': path.name,
        'shape': df.shape,
        'columnas': list(df.columns),
        'n_columnas': len(df.columns),
        'duplicados_totales': int(df.duplicated().sum()),
    }

    if {'fecha', 'departamento'}.issubset(df.columns):
        summary['duplicados_fecha_departamento'] = int(df.duplicated(subset=['fecha', 'departamento']).sum())

    if 'fecha' in df.columns:
        fechas = pd.to_datetime(df['fecha'], errors='coerce')
        summary['fecha_min'] = str(fechas.min())
        summary['fecha_max'] = str(fechas.max())

    if 'anio' in df.columns:
        anios = pd.to_numeric(df['anio'], errors='coerce')
        summary['anio_min'] = int(anios.min()) if anios.notna().any() else None
        summary['anio_max'] = int(anios.max()) if anios.notna().any() else None

    if 'departamento' in df.columns:
        summary['departamentos'] = sorted(df['departamento'].dropna().astype(str).unique().tolist())

    summary['nulos_top'] = df.isnull().sum().sort_values(ascending=False).head(10).to_dict()
    return df, summary, df.head(n_preview)


def summarize_excel(path: Path, n_preview: int = 3):
    xls = pd.ExcelFile(path)
    sheet_name = xls.sheet_names[0]
    df = pd.read_excel(path, sheet_name=sheet_name)

    summary = {
        'archivo': path.name,
        'hojas': xls.sheet_names,
        'hoja_principal': sheet_name,
        'shape': df.shape,
        'columnas': list(df.columns),
        'n_columnas': len(df.columns),
        'duplicados_totales': int(df.duplicated().sum()),
        'nulos_top': df.isnull().sum().sort_values(ascending=False).head(10).to_dict(),
    }

    year_candidates = [c for c in df.columns if 'A' in str(c) or 'anio' in str(c).lower() or 'año' in str(c).lower()]
    dept_candidates = [c for c in df.columns if 'depart' in str(c).lower()]

    if year_candidates:
        years = pd.to_numeric(df[year_candidates[0]], errors='coerce')
        summary['anio_min'] = int(years.min()) if years.notna().any() else None
        summary['anio_max'] = int(years.max()) if years.notna().any() else None

    if dept_candidates:
        summary['departamentos'] = sorted(df[dept_candidates[0]].dropna().astype(str).str.strip().unique().tolist())

    return df, summary, df.head(n_preview)


def summarize_txt(path: Path, n_lines: int = 20):
    text = path.read_text(encoding='utf-8', errors='replace')
    lines = text.splitlines()
    summary = {
        'archivo': path.name,
        'n_lineas': len(lines),
        'n_caracteres': len(text),
    }
    preview = '\n'.join(lines[:n_lines])
    return summary, preview


## Perfilado individual de cada insumo

En esta seccion se resume cada archivo clave para entender rapidamente su rol en el pipeline.


In [23]:
base_clima_df, base_clima_summary, base_clima_preview = summarize_csv(PATHS['entrada_clima_satelite'], sep=',')
pd.DataFrame([base_clima_summary]).T


,0
archivo,datos_seguro_cafe_2000_2025.csv
shape,"(628, 32)"
columnas,"[fecha, departamento, precipitation, temp_aire_C, humedad_relativa_pct, soil, def, pet, aet, GDD_cafe, NDVI, EVI, LS..."
n_columnas,32
duplicados_totales,0
duplicados_fecha_departamento,0
fecha_min,2000-01-01 00:00:00
fecha_max,2026-02-01 00:00:00
anio_min,2000
anio_max,2026


In [24]:
base_clima_preview


,fecha,departamento,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,fecha_dt,mes,anio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,indice_perdida,evento_perdida
0,2000-01,Cundinamarca,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-01-01,1,2000,NaN,NaN,-15.291502,NaN,-6.983227,-15.291502,1
1,2000-02,Cundinamarca,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-02-01,2,2000,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,-32.627589,1
2,2000-03,Cundinamarca,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-03-01,3,2000,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,-0.870595,0


In [25]:
agronet_df, agronet_summary, agronet_preview = summarize_excel(PATHS['entrada_agronet'])
pd.DataFrame([agronet_summary]).T


,0
archivo,agronet_cafe.xlsx
hojas,[detalle_agricola_municipal_muni]
hoja_principal,detalle_agricola_municipal_muni
shape,"(1483, 9)"
columnas,"[Año, Departamento, Municipio, Producto, Área Cosechada, Área Sembrada, Producción, Rendimiento, Ciclo]"
n_columnas,9
duplicados_totales,0
nulos_top,"{'Año': 0, 'Departamento': 0, 'Municipio': 0, 'Producto': 0, 'Área Cosechada': 0, 'Área Sembrada': 0, 'Producción': ..."
anio_min,2007
anio_max,2024


In [26]:
agronet_preview


,Año,Departamento,Municipio,Producto,Área Cosechada,Área Sembrada,Producción,Rendimiento,Ciclo
0,2019,RISARALDA,APIA,CAFE,3266.00,4338.00,4335.00,1.327312,PERMANENTE
1,2014,RISARALDA,APIA,CAFE,3922.46,5061.94,3399.13,0.866581,PERMANENTE
2,2007,RISARALDA,APIA,CAFE,4451.00,4851.00,7410.60,1.664929,PERMANENTE


In [27]:
base_integrada_df, base_integrada_summary, base_integrada_preview = summarize_csv(PATHS['intermedia_integrada'], sep=';')
pd.DataFrame([base_integrada_summary]).T


,0
archivo,base_datos_seguro_cafe.csv
shape,"(628, 42)"
columnas,"[fecha, departamento, mes, anio, precipitation, temp_aire_C, humedad_relativa_pct, soil, def, pet, aet, GDD_cafe, ND..."
n_columnas,42
duplicados_totales,0
duplicados_fecha_departamento,0
fecha_min,2000-01-01 00:00:00
fecha_max,2026-02-01 00:00:00
anio_min,2000
anio_max,2026


In [28]:
base_integrada_preview


,fecha,departamento,mes,anio,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,area_cosechada_ha,area_sembrada_ha,produccion_t,rendimiento_t_ha,rendimiento_medio_t_ha,produccion_media_t,factor_mensual,produccion_mensual_t,es_mes_cosecha,perdida_real_pct,perdida_real_mensual_pct,precio_ico_usd_ton,precio_productor_usd_ton
0,2000-01,Cundinamarca,1,2000,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,NaN,NaN,-15.291502,NaN,-6.983227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1900.0,1482.0
1,2000-02,Cundinamarca,2,2000,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1900.0,1482.0
2,2000-03,Cundinamarca,3,2000,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1900.0,1482.0


In [29]:
base_limpia_df, base_limpia_summary, base_limpia_preview = summarize_csv(PATHS['final_limpia'], sep=';')
pd.DataFrame([base_limpia_summary]).T


,0
archivo,cafe_clean.csv
shape,"(432, 43)"
columnas,"[fecha, departamento, mes, anio, precipitation, temp_aire_C, humedad_relativa_pct, soil, def, pet, aet, GDD_cafe, ND..."
n_columnas,43
duplicados_totales,0
duplicados_fecha_departamento,0
fecha_min,2007-01-01 00:00:00
fecha_max,2024-12-01 00:00:00
anio_min,2007
anio_max,2024


In [30]:
base_limpia_preview


,fecha,departamento,mes,anio,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,area_cosechada_ha,area_sembrada_ha,produccion_t,rendimiento_t_ha,rendimiento_medio_t_ha,produccion_media_t,factor_mensual,produccion_mensual_t,es_mes_cosecha,perdida_real_pct,perdida_real_mensual_pct,precio_ico_usd_ton,precio_productor_usd_ton,perdida_mensual_reconstruida
0,2007-01-01,Cundinamarca,1,2007,51.589846,17.711611,74.297271,678.436080,295.423041,1002.517303,707.037461,8.217083,0.671119,0.415271,24.505147,12.951798,0.053155,2.419353,0.566478,1895.307361,1086.421734,14.454071,10.320921,185.295099,2.811280,4.659527,-0.171117,3.609412,-17.437197,43017.3,48195.69,33729.14373,0.784083,0.925118,32380.09908,0.040,1349.165749,0.0,-15.245048,0.0,2750.0,2145.0,0.0
1,2007-02-01,Cundinamarca,2,2007,33.088142,17.982266,61.059155,413.344280,622.921949,1136.944986,513.898186,7.957435,0.637020,0.361616,27.230381,12.853590,0.053134,2.554727,0.589616,1895.307361,1086.421734,14.454071,10.320921,185.295099,13.614659,3.104356,12.429181,42.731016,-59.467611,43017.3,48195.69,33729.14373,0.784083,0.925118,32380.09908,0.035,1180.520031,0.0,-15.245048,0.0,2750.0,2145.0,0.0
2,2007-03-01,Cundinamarca,3,2007,150.460122,17.653585,74.953280,860.151924,4.330455,1043.557889,1039.188961,8.272950,0.463090,0.300295,26.668700,10.508166,0.039737,1.222863,0.326861,1895.307361,1086.421734,14.454071,10.320921,185.295099,-10.920556,-11.299199,-8.552304,-23.699163,5.910910,43017.3,48195.69,33729.14373,0.784083,0.925118,32380.09908,0.045,1517.811468,0.0,-15.245048,0.0,2750.0,2145.0,0.0


In [31]:
if PATHS['documentacion_txt'] is not None:
    txt_summary, txt_preview = summarize_txt(PATHS['documentacion_txt'])
    display(pd.DataFrame([txt_summary]).T)
    print(txt_preview)
else:
    print('No se encontro archivo TXT de documentacion en BASE_DE_DATOS.')


,0
archivo,VARIABLES DEL MODELO - SEGURO AGRÍCOLA.txt
n_lineas,283
n_caracteres,12929


VARIABLES DE LA BASE DE DATOS — SEGURO AGRÍCOLA INDEXADO CAFÉ
Cundinamarca y Risaralda, Colombia | 2000–2026 | Mensual
Shape: 628 filas × 42 columnas


VARIABLES DE IDENTIFICACIÓN
-----------------------------
fecha               : Año y mes (formato YYYY-MM)
departamento        : Cundinamarca o Risaralda
mes                 : Número del mes (1-12)
anio                : Año (2000-2026)


CLIMA
------
precipitation       : Precipitación acumulada del mes (mm)
                      Fuente: CHIRPS — Climate Hazards Group InfraRed
                      Precipitation with Station data (NASA/USGS)
                      Resolución: ~5km | Disponible desde: 1981


## Comparacion estructural entre las bases CSV del pipeline

Este bloque sirve para ver de manera explicita como evolucionan las bases entre entrada, intermedia y final.


In [32]:
comparacion_bases = pd.DataFrame([
    {
        'base': 'entrada_clima_satelite',
        'filas': base_clima_df.shape[0],
        'columnas': base_clima_df.shape[1],
        'fecha_min': base_clima_summary.get('fecha_min'),
        'fecha_max': base_clima_summary.get('fecha_max'),
        'anio_min': base_clima_summary.get('anio_min'),
        'anio_max': base_clima_summary.get('anio_max'),
    },
    {
        'base': 'intermedia_integrada',
        'filas': base_integrada_df.shape[0],
        'columnas': base_integrada_df.shape[1],
        'fecha_min': base_integrada_summary.get('fecha_min'),
        'fecha_max': base_integrada_summary.get('fecha_max'),
        'anio_min': base_integrada_summary.get('anio_min'),
        'anio_max': base_integrada_summary.get('anio_max'),
    },
    {
        'base': 'final_limpia',
        'filas': base_limpia_df.shape[0],
        'columnas': base_limpia_df.shape[1],
        'fecha_min': base_limpia_summary.get('fecha_min'),
        'fecha_max': base_limpia_summary.get('fecha_max'),
        'anio_min': base_limpia_summary.get('anio_min'),
        'anio_max': base_limpia_summary.get('anio_max'),
    },
])

comparacion_bases


,base,filas,columnas,fecha_min,fecha_max,anio_min,anio_max
0,entrada_clima_satelite,628,32,2000-01-01 00:00:00,2026-02-01 00:00:00,2000,2026
1,intermedia_integrada,628,42,2000-01-01 00:00:00,2026-02-01 00:00:00,2000,2026
2,final_limpia,432,43,2007-01-01 00:00:00,2024-12-01 00:00:00,2007,2024


In [33]:
solo_en_final = sorted(set(base_limpia_df.columns) - set(base_integrada_df.columns))
solo_en_intermedia = sorted(set(base_integrada_df.columns) - set(base_limpia_df.columns))
solo_en_entrada = sorted(set(base_clima_df.columns) - set(base_integrada_df.columns))
agregadas_en_intermedia = sorted(set(base_integrada_df.columns) - set(base_clima_df.columns))

comparacion_columnas = {
    'columnas_solo_en_final': solo_en_final,
    'columnas_solo_en_intermedia': solo_en_intermedia,
    'columnas_solo_en_entrada': solo_en_entrada,
    'columnas_agregadas_en_intermedia_vs_entrada': agregadas_en_intermedia,
}

comparacion_columnas


{'columnas_solo_en_final': ['perdida_mensual_reconstruida'],
 'columnas_solo_en_intermedia': [],
 'columnas_solo_en_entrada': ['evento_perdida', 'fecha_dt', 'indice_perdida'],
 'columnas_agregadas_en_intermedia_vs_entrada': ['area_cosechada_ha',
  'area_sembrada_ha',
  'es_mes_cosecha',
  'factor_mensual',
  'perdida_real_mensual_pct',
  'perdida_real_pct',
  'precio_ico_usd_ton',
  'precio_productor_usd_ton',
  'produccion_media_t',
  'produccion_mensual_t',
  'produccion_t',
  'rendimiento_medio_t_ha',
  'rendimiento_t_ha']}

## Chequeos puntuales de riesgo metodologico

Estos chequeos responden a hallazgos ya detectados en la documentacion previa del proyecto.


In [34]:
hallazgos = []

if 'factor_mensual' in base_integrada_df.columns:
    factor_sum = base_integrada_df[base_integrada_df['anio'] == 2007].groupby('departamento')['factor_mensual'].sum().to_dict()
    hallazgos.append({'chequeo': 'suma_factor_mensual_2007', 'resultado': factor_sum})

if 'perdida_mensual_reconstruida' in base_limpia_df.columns:
    desc = base_limpia_df['perdida_mensual_reconstruida'].describe().to_dict()
    hallazgos.append({'chequeo': 'distribucion_perdida_mensual_reconstruida', 'resultado': desc})

if {'anio', 'departamento', 'rendimiento_t_ha'}.issubset(base_limpia_df.columns):
    muestra_2008 = base_limpia_df[(base_limpia_df['departamento'] == 'Cundinamarca') & (base_limpia_df['anio'] == 2008)][[
        'fecha', 'departamento', 'anio', 'produccion_t', 'rendimiento_t_ha', 'perdida_real_pct'
    ]].head(3)
else:
    muestra_2008 = pd.DataFrame()

pd.DataFrame(hallazgos)


,chequeo,resultado
0,suma_factor_mensual_2007,"{'Cundinamarca': 0.95, 'Risaralda': 0.95}"
1,distribucion_perdida_mensual_reconstruida,"{'count': 432.0, 'mean': -44.458333333451634, 'std': 44.63956957652873, 'min': -95.159480531151, '25%': -89.87048205..."


In [35]:
muestra_2008


,fecha,departamento,anio,produccion_t,rendimiento_t_ha,perdida_real_pct
12,2008-01-01,Cundinamarca,2008,35732.353882,0.818923,-11.479067
13,2008-02-01,Cundinamarca,2008,35732.353882,0.818923,-11.479067
14,2008-03-01,Cundinamarca,2008,35732.353882,0.818923,-11.479067


## Resumen interpretativo

Este bloque produce un resumen automatico para dejar trazabilidad minima de la auditoria.


In [36]:
resumen = {
    'entrada_clima_satelite': {
        'shape': base_clima_df.shape,
        'rol': 'base de entrada de clima, satelite, terreno y anomalias',
    },
    'entrada_agronet': {
        'shape': agronet_df.shape,
        'rol': 'fuente cruda productiva municipal para agregar a departamento-anio',
    },
    'intermedia_integrada': {
        'shape': base_integrada_df.shape,
        'rol': 'base mensual integrada con clima, satelite, Agronet distribuido y precios',
    },
    'final_limpia': {
        'shape': base_limpia_df.shape,
        'rol': 'base limpia exploratoria recortada a 2007-2024, util para referencia pero no para heredarla sin revision',
    },
    'hallazgos_clave': [
        'la base intermedia contiene la integracion principal del pipeline',
        'la base final agrega la columna perdida_mensual_reconstruida',
        'el factor_mensual debe revisarse si no suma 1 por departamento-anio',
        'la fuente Agronet local permite reconstruir la base anual desde origen',
    ],
}

print(json.dumps(resumen, indent=2, ensure_ascii=False))


{
  "entrada_clima_satelite": {
    "shape": [
      628,
      32
    ],
    "rol": "base de entrada de clima, satelite, terreno y anomalias"
  },
  "entrada_agronet": {
    "shape": [
      1483,
      9
    ],
    "rol": "fuente cruda productiva municipal para agregar a departamento-anio"
  },
  "intermedia_integrada": {
    "shape": [
      628,
      42
    ],
    "rol": "base mensual integrada con clima, satelite, Agronet distribuido y precios"
  },
  "final_limpia": {
    "shape": [
      432,
      43
    ],
    "rol": "base limpia exploratoria recortada a 2007-2024, util para referencia pero no para heredarla sin revision"
  },
  "hallazgos_clave": [
    "la base intermedia contiene la integracion principal del pipeline",
    "la base final agrega la columna perdida_mensual_reconstruida",
    "el factor_mensual debe revisarse si no suma 1 por departamento-anio",
    "la fuente Agronet local permite reconstruir la base anual desde origen"
  ]
}


## Decision operativa sugerida para el siguiente notebook

Si esta auditoria confirma que los archivos existen y son consistentes con lo esperado, el siguiente paso recomendado es construir el notebook `02_agronet_anual_limpio.ipynb` para:

- agregar Agronet municipal a `departamento-anio`
- revisar y corregir Cundinamarca 2008
- recalcular rendimiento y variables derivadas
- exportar una base anual intermedia confiable

Ese paso es critico porque define la capa de productividad observada que luego alimentara el dataset principal de modelado.
